# 🚀 YOLO26 Image Classification - 10 Product Training

This notebook trains a **YOLO26-cls** model to classify **10 snack products** using transfer learning.

### Products:
1. Balaji Banana Wafer Mast Mari
2. Balaji Crunchem Simply Salted
3. Balaji Gippi Tornado
4. Chana Dal Balaji
5. Funne Balaji
6. Gokul Nylon Gathiya
7. Gopal Masala Cup
8. Mug Dal Balaji
9. Sticks Gopal
10. Wheels Balaji

### Steps:
1. Install Ultralytics
2. Mount Google Drive & load dataset
3. Prepare train/val split
4. Train YOLO26-cls model
5. Evaluate results
6. Test predictions
7. Export model

---
⚠️ **Before running:** Make sure to set Runtime → Change runtime type → **T4 GPU**

## Step 1: Install Ultralytics

In [ ]:
!pip install ultralytics -q

import ultralytics
ultralytics.checks()
print(f"\n✅ Ultralytics {ultralytics.__version__} installed successfully!")

## Step 2: Mount Google Drive & Upload Dataset

### Instructions:
1. **Zip** your `SSIP_P_3_PHOTOES_COLLECTION` folder (the one with 10 product subfolders)
2. **Upload** the zip file to your Google Drive (root or any folder)
3. **Update the path** in the next cell if you placed it somewhere other than root

Your zip should contain this structure:
```
SSIP_P_3_PHOTOES_COLLECTION/
├── Balaji Banana Wafer Mast Mari/
│   ├── img1.jpg, img2.jpg, ...
├── Balaji Crunchem Simply Salted/
│   ├── img1.jpg, img2.jpg, ...
└── ... (all 10 product folders)
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive mounted!")

In [ ]:
import zipfile
import os

# ============================================================
# 👇 UPDATE THIS PATH to where you uploaded the zip in Drive
# ============================================================
ZIP_PATH = "/content/drive/MyDrive/SSIP_P_3_PHOTOES_COLLECTION.zip"

# Extract to /content/
EXTRACT_DIR = "/content/raw_dataset"

if os.path.exists(ZIP_PATH):
    print(f"📦 Found zip file: {ZIP_PATH}")
    print("📂 Extracting...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)
    print("✅ Extraction complete!")
else:
    print(f"❌ Zip file not found at: {ZIP_PATH}")
    print("   Please update the ZIP_PATH variable above.")
    print("\n   Example paths:")
    print('   ZIP_PATH = "/content/drive/MyDrive/SSIP_P_3_PHOTOES_COLLECTION.zip"')
    print('   ZIP_PATH = "/content/drive/MyDrive/SSIP/dataset.zip"')

# Find the actual dataset root (folder containing the 10 product subfolders)
DATASET_ROOT = None
for root, dirs, files in os.walk(EXTRACT_DIR):
    # Look for a directory that has multiple subdirectories with images
    subdirs_with_images = 0
    for d in dirs:
        subdir_path = os.path.join(root, d)
        has_images = any(
            f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.webp'))
            for f in os.listdir(subdir_path)
            if os.path.isfile(os.path.join(subdir_path, f))
        )
        if has_images:
            subdirs_with_images += 1
    if subdirs_with_images >= 5:  # At least 5 product folders found
        DATASET_ROOT = root
        break

if DATASET_ROOT:
    print(f"\n📁 Dataset root found: {DATASET_ROOT}")
    product_folders = [d for d in sorted(os.listdir(DATASET_ROOT))
                       if os.path.isdir(os.path.join(DATASET_ROOT, d))]
    print(f"\n🏷️  Found {len(product_folders)} product classes:")
    for i, folder in enumerate(product_folders, 1):
        count = len([f for f in os.listdir(os.path.join(DATASET_ROOT, folder))
                     if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.webp'))])
        print(f"   {i:2d}. {folder} ({count} images)")
else:
    print("❌ Could not find product folders. Check your zip structure.")

## Step 3: Prepare Dataset (Train/Val Split)

YOLO classification requires this structure:
```
dataset/
├── train/
│   ├── Class_1/
│   ├── Class_2/
│   └── ...
└── val/
    ├── Class_1/
    ├── Class_2/
    └── ...
```

We'll do an **80/20 split** (80% training, 20% validation).

In [ ]:
import shutil
import random

# Set random seed for reproducibility
random.seed(42)

# Output dataset directory
DATASET_DIR = "/content/product_dataset"
TRAIN_DIR = os.path.join(DATASET_DIR, "train")
VAL_DIR = os.path.join(DATASET_DIR, "val")

# Clean up if exists
if os.path.exists(DATASET_DIR):
    shutil.rmtree(DATASET_DIR)

# Split ratio
TRAIN_RATIO = 0.80

total_train = 0
total_val = 0

print("📊 Splitting dataset (80% train / 20% val):\n")
print(f"{'Product':<40} {'Train':>6} {'Val':>6} {'Total':>6}")
print("-" * 62)

for folder in sorted(os.listdir(DATASET_ROOT)):
    folder_path = os.path.join(DATASET_ROOT, folder)
    if not os.path.isdir(folder_path):
        continue

    # Skip the flattened folder if it exists
    if folder == "All_Products_Flattened":
        continue

    # Sanitize folder name (replace spaces with underscores)
    clean_name = folder.strip().replace(" ", "_")

    # Get all image files
    images = [f for f in os.listdir(folder_path)
              if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.webp'))]
    random.shuffle(images)

    # Split
    split_idx = int(len(images) * TRAIN_RATIO)
    train_images = images[:split_idx]
    val_images = images[split_idx:]

    # Create directories
    train_class_dir = os.path.join(TRAIN_DIR, clean_name)
    val_class_dir = os.path.join(VAL_DIR, clean_name)
    os.makedirs(train_class_dir, exist_ok=True)
    os.makedirs(val_class_dir, exist_ok=True)

    # Copy train images
    for img in train_images:
        shutil.copy2(os.path.join(folder_path, img), os.path.join(train_class_dir, img))

    # Copy val images
    for img in val_images:
        shutil.copy2(os.path.join(folder_path, img), os.path.join(val_class_dir, img))

    total_train += len(train_images)
    total_val += len(val_images)

    print(f"{clean_name:<40} {len(train_images):>6} {len(val_images):>6} {len(images):>6}")

print("-" * 62)
print(f"{'TOTAL':<40} {total_train:>6} {total_val:>6} {total_train + total_val:>6}")
print(f"\n✅ Dataset prepared at: {DATASET_DIR}")

## Step 4: Train YOLO26-cls Model 🏋️

We use **`yolo26n-cls.pt`** (nano) — the smallest and fastest model, ideal for fine-tuning on a small dataset with transfer learning from ImageNet.

| Parameter | Value | Why |
|-----------|-------|-----|
| Model | `yolo26n-cls.pt` | Small model, prevents overfitting on limited data |
| Epochs | 50 | Sufficient for fine-tuning |
| Image Size | 224 | Standard classification size |
| Batch Size | 32 | Good balance for T4 GPU |
| Optimizer | auto | Uses YOLO26's MuSGD optimizer |

In [ ]:
from ultralytics import YOLO

# Load pretrained YOLO26 classification model (nano)
model = YOLO("yolo26n-cls.pt")

# Train the model
results = model.train(
    data=DATASET_DIR,       # Path to our prepared dataset
    epochs=50,              # Number of training epochs
    imgsz=224,              # Image size for classification
    batch=32,               # Batch size (T4 GPU can handle this)
    patience=10,            # Early stopping patience
    pretrained=True,        # Use ImageNet pretrained weights
    device=0,               # Use GPU
    workers=2,              # Data loading workers
    project="/content/yolo26_products",  # Save results here
    name="train",           # Run name
    exist_ok=True,          # Overwrite if exists
    verbose=True            # Show training progress
)

print("\n" + "="*50)
print("✅ TRAINING COMPLETE!")
print("="*50)

## Step 5: View Training Results 📊

In [ ]:
from IPython.display import display, Image
import glob

results_dir = "/content/yolo26_products/train"

# Display training results
print("📈 Training Results:\n")

# Show results plot
results_img = os.path.join(results_dir, "results.png")
if os.path.exists(results_img):
    print("Training Curves:")
    display(Image(filename=results_img, width=800))

# Show confusion matrix
cm_img = os.path.join(results_dir, "confusion_matrix.png")
if os.path.exists(cm_img):
    print("\nConfusion Matrix:")
    display(Image(filename=cm_img, width=600))

# Show normalized confusion matrix
cm_norm_img = os.path.join(results_dir, "confusion_matrix_normalized.png")
if os.path.exists(cm_norm_img):
    print("\nNormalized Confusion Matrix:")
    display(Image(filename=cm_norm_img, width=600))

## Step 6: Validate Model & Check Accuracy ✅

In [ ]:
from ultralytics import YOLO

# Load the best trained model
best_model = YOLO("/content/yolo26_products/train/weights/best.pt")

# Run validation
metrics = best_model.val()

print("\n" + "="*50)
print("📊 VALIDATION RESULTS")
print("="*50)
print(f"Top-1 Accuracy: {metrics.top1:.4f} ({metrics.top1*100:.2f}%)")
print(f"Top-5 Accuracy: {metrics.top5:.4f} ({metrics.top5*100:.2f}%)")
print("="*50)

## Step 7: Test Predictions on Sample Images 🔍

In [ ]:
from ultralytics import YOLO
from IPython.display import display, Image as IPyImage
from PIL import Image as PILImage
import matplotlib.pyplot as plt
import random

# Load best model
best_model = YOLO("/content/yolo26_products/train/weights/best.pt")

# Collect sample images from validation set (1 per class)
val_dir = os.path.join(DATASET_DIR, "val")
sample_images = []

for class_folder in sorted(os.listdir(val_dir)):
    class_path = os.path.join(val_dir, class_folder)
    if os.path.isdir(class_path):
        images = [f for f in os.listdir(class_path)
                  if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        if images:
            img_path = os.path.join(class_path, random.choice(images))
            sample_images.append((img_path, class_folder))

# Run predictions and display
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
fig.suptitle("YOLO26 Product Classification - Predictions", fontsize=16, fontweight='bold')

for idx, (img_path, true_label) in enumerate(sample_images[:10]):
    row = idx // 5
    col = idx % 5
    ax = axes[row][col]

    # Run prediction
    result = best_model(img_path, verbose=False)[0]
    predicted_class = result.names[result.probs.top1]
    confidence = result.probs.top1conf.item()

    # Display image
    img = PILImage.open(img_path)
    ax.imshow(img)

    # Color based on correct/incorrect
    is_correct = predicted_class == true_label
    color = 'green' if is_correct else 'red'
    symbol = '✅' if is_correct else '❌'

    ax.set_title(f"{symbol} {predicted_class}\n({confidence*100:.1f}%)",
                 fontsize=8, color=color, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.savefig("/content/yolo26_products/predictions_sample.png", dpi=150, bbox_inches='tight')
plt.show()
print("\n✅ Prediction results saved!")

## Step 8: Export Model 📦

Export the trained model to **ONNX** format for deployment.

In [ ]:
from ultralytics import YOLO

# Load best model
best_model = YOLO("/content/yolo26_products/train/weights/best.pt")

# Export to ONNX
onnx_path = best_model.export(format="onnx", imgsz=224)
print(f"\n✅ Model exported to ONNX: {onnx_path}")

# Also export to TorchScript
torchscript_path = best_model.export(format="torchscript", imgsz=224)
print(f"✅ Model exported to TorchScript: {torchscript_path}")

## Step 9: Save Model to Google Drive & Download 💾

Copy all trained model files to your Google Drive for safekeeping.

In [ ]:
import shutil
import glob

# Save to Google Drive
DRIVE_SAVE_DIR = "/content/drive/MyDrive/YOLO26_Product_Model"
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

# Copy best weights
shutil.copy2("/content/yolo26_products/train/weights/best.pt",
             os.path.join(DRIVE_SAVE_DIR, "best.pt"))
print("✅ best.pt saved to Google Drive")

# Copy last weights
shutil.copy2("/content/yolo26_products/train/weights/last.pt",
             os.path.join(DRIVE_SAVE_DIR, "last.pt"))
print("✅ last.pt saved to Google Drive")

# Copy ONNX model if exists
onnx_files = glob.glob("/content/yolo26_products/train/weights/*.onnx")
for f in onnx_files:
    shutil.copy2(f, os.path.join(DRIVE_SAVE_DIR, os.path.basename(f)))
    print(f"✅ {os.path.basename(f)} saved to Google Drive")

# Copy results
for fname in ["results.csv", "results.png", "confusion_matrix.png",
              "confusion_matrix_normalized.png"]:
    fpath = os.path.join("/content/yolo26_products/train", fname)
    if os.path.exists(fpath):
        shutil.copy2(fpath, os.path.join(DRIVE_SAVE_DIR, fname))

print(f"\n📁 All files saved to: {DRIVE_SAVE_DIR}")
print("\n📋 Files saved:")
for f in sorted(os.listdir(DRIVE_SAVE_DIR)):
    size_mb = os.path.getsize(os.path.join(DRIVE_SAVE_DIR, f)) / (1024*1024)
    print(f"   📄 {f} ({size_mb:.2f} MB)")

## 🎯 How to Use Your Trained Model Later

```python
from ultralytics import YOLO

# Load your trained model
model = YOLO("best.pt")

# Predict on a new image
results = model("path/to/new_product_image.jpg")

# Get the prediction
for result in results:
    predicted_class = result.names[result.probs.top1]
    confidence = result.probs.top1conf.item()
    print(f"Product: {predicted_class}")
    print(f"Confidence: {confidence*100:.2f}%")
```